<a href="https://colab.research.google.com/github/josephasal/cosmo_inference/blob/4-parameters/4D_mcmc/4Dmcmc_functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This file has all the code for the functions:

* Distance modulus + fitting function
* log likelihood and prior
* basic & adaptive mcmc
* Convergence via Gellman Rubin diagnostic
* Autocorrelation
* Effective sample size

except that it is now in 4 dimensions to allow for the $\Omega_k$ and $\Omega_{\Lambda}$ to be inferred as well

Need to re write distance modulus, log prior, log likelihood and both basic and adaptive mcmc code

# Distance modulus

In [1]:
import numpy as np
from scipy.integrate import quad

def calculate_distance_modulus(z, omega_m, omega_k, omega_lambda, h):
  """
  Calculates the dimensionless distance modulus using

  inputs:
  - z: redshift
  - omega_m: density matter parameter
  - omega_k: curvature parameter
  - omega_lambda: dark energy parameter
  - h: dimensionless hubble contant

  outputs: theoretical distance modulus (mu_theoretical)

  """
  c = 299792.458 # speed of light in km/s
  H0 = 100 * h   #Hubble constant in km/s/Mpc

  #Expansion rate
  def expansion_rate(z, omega_m, omega_k, omega_lambda):
    return np.sqrt(omega_m * (1+z)**3 + omega_k * (1+z)**2 + omega_lambda)

  #Comoving distance, using scipy integrate, couldnt find a well adopted fitting formula for non flat universe
  def integrand(z, omega_m, omega_k, omega_lambda):
    return 1/ expansion_rate(z, omega_m, omega_k, omega_lambda)

  result, error = quad(integrand, 0, z, args = (omega_m, omega_k, omega_lambda)) #the omegas are fixed for the integration each cycle
  d_comoving = (c/H0) * result

  # Transverse co moving distance
  #based on curvature of universe this can be different

  #Flat universe
  #if omega_k = 0:
  if np.abs(omega_k) < 1e-6:
    d_M = d_comoving

  #Open universe
  elif omega_k > 0:
    d_M = (c / H0 * np.sqrt(omega_k)) * np.sinh(np.sqrt(omega_k) * ((H0 * d_comoving) / c))

  #Closed universe
  else: d_M = (c/ H0 * np.sqrt(np.abs(omega_k))) * np.sin(np.sqrt(omega_k) * (H0 * d_comving)/c)


  #Luminosty distance
  d_L = (1+z) * d_M

  #Distance modulus
  theoretical_mu = 25 - 5*np.log10(h) + 5*np.log10(d_L)

  return theoretical_mu

# Likelihood and Prior

In [3]:
#Likelihood function, standard gaussian function

def log_likelihood(mu_obs, mu_model, sigma_mu):

  """
  Computes the logged likelihood for the sample and observed distance modulus

  inputs:
    - mu_obs: observed mu (mu from the data)
    - mu_model: theoretical mu from the model
    - sigma_mu: standard deviation of the observed mu (uncertainty)

  outputs:
    - log likelihood

  """
  return -0.5 * np.sum((mu_obs - mu_model)**2/sigma_mu**2)



# Defining the prior as a function
def log_prior(params):
  """
  Function that sets a uniform prior of the omega parameters and h, based off the planck data

  """
  omega_m, omega_k, omega_lambda, h = params
  if 0.1 < omega_m < 0.5 and -0.2 < omega_k < 0.2 and 0.6 < omega_lambda < 0.8 and 0.4 < h < 0.9:
    return 0.0 #log of 1 is 0

  else:
    return -np.inf #acceptance probability is 0 if not in the priors upper and lower bounds


# Basic MCMC